In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q segmentation-models-pytorch albumentations lightgbm shap scikit-image timm opencv-python-headless joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.4 MB/s eta 0:00:00


In [ ]:
!rm -rf /content/forensic_project
!mkdir -p /content/forensic_project

!unzip -o -q /content/drive/MyDrive/forensic_uploads/forensic_project_code_fixed_flat.zip -d /content/forensic_project

!mkdir -p /content/drive/MyDrive/forensic_project
!cp /content/forensic_project/*.py /content/drive/MyDrive/forensic_project/
!cp /content/forensic_project/README.txt /content/drive/MyDrive/forensic_project/ 2>/dev/null || true

!ls -al /content/drive/MyDrive/forensic_project

total 67
-rw------- 1 root root  3737 Jun  8 09:14 README.txt
-rw------- 1 root root  1164 Jun  8 09:14 step00_setup.py
-rw------- 1 root root  7382 Jun  8 09:14 step01_data_prep.py
-rw------- 1 root root 13755 Jun  8 09:14 step02_train_stage1.py
-rw------- 1 root root  6431 Jun  8 09:14 step03_heatmap_features.py
-rw------- 1 root root  7962 Jun  8 09:14 step04_train_stage2.py
-rw------- 1 root root 10871 Jun  8 09:14 step05_evaluate.py
-rw------- 1 root root 14577 Jun  8 09:14 utils.py


In [ ]:
!ls -lh /content/drive/MyDrive/forensic_project/checkpoints

ls: cannot access '/content/drive/MyDrive/forensic_project/checkpoints': No such file or directory


In [ ]:
!mkdir -p /content/zips

!cp /content/drive/MyDrive/forensic_uploads/defacto_15000_probe.zip /content/zips/
!cp /content/drive/MyDrive/forensic_uploads/coco_real_15000.zip /content/zips/

!python /content/drive/MyDrive/forensic_project/step01_data_prep.py --clean \
  --defacto_zip /content/zips/defacto_15000_probe.zip \
  --coco_zip /content/zips/coco_real_15000.zip

^C
^C
압축 해제: /content/zips/defacto_15000_probe.zip -> /content/datasets
Traceback (most recent call last):
  File "/content/drive/MyDrive/forensic_project/step01_data_prep.py", line 188, in <module>
    main()
  File "/content/drive/MyDrive/forensic_project/step01_data_prep.py", line 164, in main
    unzip_clean(args.defacto_zip, extract_dir)
  File "/content/drive/MyDrive/forensic_project/step01_data_prep.py", line 46, in unzip_clean
    with zipfile.ZipFile(zip_path, "r") as zf:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1370, in __init__
    self._RealGetContents()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1437, in _RealGetContents
    raise BadZipFile("File is not a zip file")
zipfile.BadZipFile: File is not a zip file


In [ ]:
%%writefile /content/drive/MyDrive/forensic_project/step02_run_one.py
import os
import sys
import json
import argparse
import numpy as np
import pandas as pd
import torch

BASE_DIR = "/content/drive/MyDrive/forensic_project"
sys.path.insert(0, BASE_DIR)

import step02_train_stage1 as s2


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--run_id", type=int, required=True)
    args = parser.parse_args()

    if args.run_id not in [0, 1, 2]:
        raise ValueError("run_id는 0, 1, 2 중 하나여야 합니다.")

    seed = s2.CFG["seeds"][args.run_id]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"device: {device}")
    print(f"Run {args.run_id + 1}/3만 실행합니다. seed={seed}")

    df = pd.read_csv(s2.CSV_PATH)
    train_df = df[df["split"] == "train"].reset_index(drop=True)

    missing_img = (~df["image_path"].map(os.path.exists)).sum()
    missing_msk = (~df["mask_path"].map(os.path.exists)).sum()

    if missing_img or missing_msk:
        raise FileNotFoundError(f"경로 오류: missing images={missing_img}, missing masks={missing_msk}")

    groups = train_df["group_id"].unique()
    rng = np.random.RandomState(seed)
    rng.shuffle(groups)

    split_n = int(len(groups) * (1 - s2.CFG["inner_val_ratio"]))
    inner_train_grps = set(groups[:split_n])
    inner_val_grps = set(groups[split_n:])

    inner_train_df = train_df[train_df["group_id"].isin(inner_train_grps)]
    inner_val_df = train_df[train_df["group_id"].isin(inner_val_grps)]

    best_dice, ckpt_path, epoch_log = s2.train_single_run(
        inner_train_df,
        inner_val_df,
        run_id=args.run_id,
        seed=seed,
        device=device
    )

    os.makedirs(f"{BASE_DIR}/results", exist_ok=True)
    log_path = f"{BASE_DIR}/results/stage1_run{args.run_id}_log.json"

    log = {
        "run": args.run_id,
        "seed": seed,
        "best_val_dice": float(best_dice),
        "ckpt_path": ckpt_path,
        "epoch_log": epoch_log
    }

    with open(log_path, "w") as f:
        json.dump(log, f, indent=2, default=str)

    print(f"Run {args.run_id + 1}/3 완료")
    print(f"best dice: {best_dice}")
    print(f"checkpoint: {ckpt_path}")
    print(f"log: {log_path}")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/forensic_project/step02_run_one.py


In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step02_run_one.py --run_id 2

device: cuda
Run 3/3만 실행합니다. seed=456
Traceback (most recent call last):
  File "/content/drive/MyDrive/forensic_project/step02_run_one.py", line 78, in <module>
    main()
  File "/content/drive/MyDrive/forensic_project/step02_run_one.py", line 29, in main
    df = pd.read_csv(s2.CSV_PATH)
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/re

In [ ]:
!ls -lh /content/drive/MyDrive/forensic_project/checkpoints

total 0


In [ ]:
%%writefile /content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py
import os
import sys
import json
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

BASE_DIR = "/content/drive/MyDrive/forensic_project"
sys.path.insert(0, BASE_DIR)

from utils import ForensicDataset, compute_stage1_metrics
import step02_train_stage1 as s2


@torch.no_grad()
def evaluate_ckpt(ckpt_path, val_df, device):
    model = s2.build_model().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    val_ds = ForensicDataset(val_df, mode="val", target_size=s2.CFG["target_size"])
    val_dl = DataLoader(
        val_ds,
        batch_size=s2.CFG["batch_size"],
        shuffle=False,
        num_workers=s2.CFG["num_workers"],
        pin_memory=True
    )

    metrics = {
        "iou": [],
        "dice": [],
        "precision": [],
        "recall": [],
        "fp_ratio": []
    }

    for images, masks, labels in val_dl:
        images = images.to(device)
        preds = torch.sigmoid(model(images)).cpu().numpy()
        gts = masks.numpy()
        labs = labels.numpy()

        for pred, gt, lab in zip(preds, gts, labs):
            m = compute_stage1_metrics(pred[0], gt[0])

            if lab == 1 and gt[0].sum() > 0:
                metrics["iou"].append(m["iou"])
                metrics["dice"].append(m["dice"])
                metrics["precision"].append(m["precision"])
                metrics["recall"].append(m["recall"])

            if lab == 0 and m["fp_area_ratio"] is not None:
                metrics["fp_ratio"].append(m["fp_area_ratio"])

    def mean(x):
        return float(np.mean(x)) if len(x) > 0 else 0.0

    return {k: mean(v) for k, v in metrics.items()}


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    df = pd.read_csv(f"{BASE_DIR}/data/dataset.csv")
    val_df = df[df["split"] == "validation"].reset_index(drop=True)

    ckpts = sorted(glob.glob(f"{BASE_DIR}/checkpoints/run*_best.pth"))

    if len(ckpts) == 0:
        raise RuntimeError("checkpoint가 없습니다.")

    print("찾은 checkpoint:")
    for c in ckpts:
        print(c)

    results = []

    for ckpt in ckpts:
        print(f"\n평가 중: {ckpt}")
        val_metrics = evaluate_ckpt(ckpt, val_df, device)
        print(val_metrics)

        results.append({
            "ckpt_path": ckpt,
            "val_metrics": val_metrics,
            "external_val_dice": val_metrics["dice"]
        })

    best = max(results, key=lambda x: x["external_val_dice"])

    log = {
        "note": "Checkpoints were trained separately because Colab quota interrupted repeated hold-out training.",
        "available_checkpoints": results,
        "best_ckpt_path": best["ckpt_path"],
        "best_overall_dice": best["external_val_dice"],
        "mean_dice": float(np.mean([r["external_val_dice"] for r in results])),
        "std_dice": float(np.std([r["external_val_dice"] for r in results])),
        "val_metrics": best["val_metrics"]
    }

    out_path = f"{BASE_DIR}/results/stage1_training_log.json"
    os.makedirs(f"{BASE_DIR}/results", exist_ok=True)

    with open(out_path, "w") as f:
        json.dump(log, f, indent=2, default=str)

    print("\n최종 선택 checkpoint:", best["ckpt_path"])
    print("stage1_training_log 저장:", out_path)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py


In [ ]:
!python -u /content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py

device: cuda
Traceback (most recent call last):
  File "/content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py", line 116, in <module>
    main()
  File "/content/drive/MyDrive/forensic_project/step02_finalize_from_ckpts.py", line 68, in main
    df = pd.read_csv(f"{BASE_DIR}/data/dataset.csv")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3